<a href="https://colab.research.google.com/github/SaliElloh/modesty-project/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# config
IMAGES_DIR = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/images/'
LABELS_CSV = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/training_labels.csv'
MODEL_SAVE = '/content/drive/MyDrive/PersonalProjects/modesty/models/modesty_classifier.pth'
BATCH_SIZE = 32
EPOCHS= 15
LR = 1e-4

In [4]:
# load and balance data
df = pd.read_csv(LABELS_CSV)

modest_df = df[df['label'] == 'modest']
immodest_df = df[df['label'] == 'immodest'].sample(n=len(modest_df), random_state=42)

# combined two dataframes
balanced_df = pd.concat([modest_df, immodest_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced dataset:")
print(f"  Modest:   {(balanced_df['label']=='modest').sum()}")
print(f"  Immodest: {(balanced_df['label']=='immodest').sum()}")
print(f"  Total:    {len(balanced_df)}")



Balanced dataset:
  Modest:   6014
  Immodest: 6014
  Total:    12028


Splitting data into training and testing

In [5]:
train_df , temp_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df['label'],
    random_state=42)


val_df , test_df = train_test_split(temp_df,
                                 test_size=0.5,
                                 stratify=temp_df['label'],
                                    random_state=42)


print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


  Train: 9622 | Val: 1203 | Test: 1203


In [6]:
test_df.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/test_labels.csv', index=False)

### Create Model

In [13]:
class ModestyDataset(Dataset):
  def __init__(self, df, images_dir, transform=None):
    self.df = df.reset_index(drop=True)
    self.images_dir = images_dir
    self.transform = transform
    self.missing = 0


  def __len__(self):
    return len(self.df)


  def __getitem__(self, idx):
      row      = self.df.iloc[idx]
      img_path = os.path.join(self.images_dir, row['img_name'])

      try:
          img = Image.open(img_path).convert('RGB')
      except (FileNotFoundError, OSError): # Catch both FileNotFoundError and OSError
          self.missing += 1
          img = Image.new('RGB', (224, 224), (200, 200, 200))

      if self.transform:
          img = self.transform(img)

      label = int(row['label_binary'])
      return img, label, row['img_name']

# transform

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = ModestyDataset(train_df, IMAGES_DIR, train_transform)
val_ds   = ModestyDataset(val_df,   IMAGES_DIR, val_transform)
test_ds  = ModestyDataset(test_df,  IMAGES_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [14]:
# model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze early layers — keep general visual features
for param in list(model.parameters())[:-20]:
    param.requires_grad = False

# Binary classification head
num_features    = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, 1)
    # No sigmoid here — BCEWithLogitsLoss handles it internally
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"\nTraining on: {device}")




Training on: cuda


In [15]:
pos_weight = torch.tensor([1.0]).to(device)  # balanced so weight is 1.0
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)


In [10]:
import torch
print(torch.cuda.is_available())        # True = GPU available
print(torch.cuda.get_device_name(0))    # tells you which GPU

True
Tesla T4


In [ ]:
# training
best_val_acc = 0
os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)

for epoch in range(EPOCHS):

    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs   = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    val_preds_all, val_labels_all = [], []

    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs   = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(imgs)

            preds       = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
            val_preds_all.extend(preds.cpu().squeeze(1).tolist())
            val_labels_all.extend(labels.cpu().squeeze(1).tolist())

    train_acc = train_correct / train_total
    val_acc   = val_correct   / val_total
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {train_loss/len(train_loader):.3f} | "
          f"Train acc: {train_acc:.3f} | "
          f"Val acc: {val_acc:.3f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE)
        print(f"  ✓ Saved (val acc: {val_acc:.3f})")

print(f"\nBest val accuracy: {best_val_acc:.3f}")
print(f"Missing images during training: {train_ds.missing + val_ds.missing}")


## Testing
